# PICKO Research · NB2 — **Depth**: is parameter extraction harder with more parameters?

40 full-schema tools can't all be offered at once (token limit), and a single model gives one
unreplicated score per tool. Instead we **repeatedly sample a small, bucket-balanced set** (tools from
every param-count bucket, sized to fit the encoder), finetune, and measure argument extraction — over
several iterations — so each bucket gets many measurements and we can show **error bars**.

*Run & forget:* each iteration trains once to Drive; a restart **skips finished iterations** and keeps
the collected per-tool rows in `picko_out/depth_results.json`.

## 0 · Colab quick-start (GPU) — run & forget, restart-safe

**On Colab first: Runtime → Change runtime type → GPU (L4 recommended; T4/A100 also fine).**
This cell clones the repo, pins the exact JAX/Flax, mounts Drive, and points **both** the data (in) and
the checkpoints+results (out) at your **`MyDrive/picko/`** folder — so a runtime restart loses nothing.

**Prerequisite (one-time):** `picko_balanced.jsonl` must be in `MyDrive/picko/`. **Running locally?** This
cell is a no-op — skip to cell 1.

In [ ]:
# --- Colab bootstrap (safe to re-run; no-op locally) ---
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("/content/picko"):
        !git clone -b hadar-work https://github.com/HadarBit/picko.git /content/picko
    %pip install -q "jax[cuda12]==0.10.2" "jaxlib==0.10.2" "flax==0.12.8"
    sys.path.insert(0, "/content/picko")
    from google.colab import drive; drive.mount("/content/drive")
    import shutil
    DRIVE = "/content/drive/MyDrive/picko"                      # <- everything lives here
    os.environ["PICKO_OUT_DIR"] = f"{DRIVE}/picko_out"          # checkpoints + results (durable)
    os.environ["PICKO_LOG"]     = f"{DRIVE}/picko_out/run.log"  # durable log across restarts
    os.makedirs(os.environ["PICKO_OUT_DIR"], exist_ok=True)
    dst = "/content/picko/data/picko_balanced.jsonl"
    if not os.path.exists(dst):
        cands = [f"{DRIVE}/picko_balanced.jsonl", "/content/drive/MyDrive/picko_balanced.jsonl"]
        src = next((c for c in cands if os.path.exists(c)), None)
        if src is None:
            have = os.listdir(DRIVE) if os.path.isdir(DRIVE) else "(MyDrive/picko not found)"
            raise FileNotFoundError(
                "picko_balanced.jsonl not found. Upload it to MyDrive/picko/. "
                f"Currently in {DRIVE}: {have}")
        os.makedirs(os.path.dirname(dst), exist_ok=True); shutil.copy(src, dst)
        print("copied data from", src)
    import jax
    print("GPU:");
    !nvidia-smi -L
    print("jax devices:", jax.devices())
    _plat = jax.devices()[0].platform
    assert _plat == "gpu", (
        f"JAX is running on '{_plat}', NOT the GPU — every finetune/eval will be ~30x slower "
        "(hours instead of minutes). FIX: Runtime > Change runtime type > GPU (L4), then "
        "Runtime > Restart session, and re-run this cell. If a GPU IS selected but this still "
        "fails, the CUDA plugin didn't load — re-run the %pip line above, then restart.")
    print("bootstrap OK · GPU active · data =", dst, "· OUT_DIR =", os.environ["PICKO_OUT_DIR"])
else:
    print("Not on Colab — running locally (CPU).")

## 1 · Setup & data overview

In [ ]:
# ensure the repo root is importable (works from notebooks/research/, Colab, etc.)
import os, sys
_here = os.path.abspath(os.getcwd())
for _ in range(6):
    if os.path.exists(os.path.join(_here, "scripts", "picko_research.py")): break
    _here = os.path.dirname(_here)
if os.path.isdir("/content/picko"): _here = "/content/picko"
if _here not in sys.path: sys.path.insert(0, _here)

from scripts.picko_research import *
import json, time
import pandas as pd, numpy as np, matplotlib.pyplot as plt
try:
    import seaborn as sns; sns.set_theme(style="whitegrid")
except Exception:
    sns = None
from tqdm.auto import tqdm

cat, tok, raw, FOCUS, OUT_DIR = load_context()
env_report(OUT_DIR)   # jax devices + is OUT_DIR durable (Drive)?

### The 40 focus tools\nOne row per tool, with its family, category and **parameter count / bucket**.

In [ ]:
display(tools_dataframe(cat, FOCUS))

### All examples for these 40 tools\nOne row per training example (query → gold tool), tagged with the gold tool's **param bucket**.

In [ ]:
ex_df = examples_dataframe(cat, raw, FOCUS)
print("examples:", ex_df.shape[0], "| per param bucket:", ex_df["param_bucket"].value_counts().to_dict())
display(ex_df.head(10))

## 2 · Configure the repeated sampling\nEach iteration draws `TOOLS_PER_BUCKET` tools from **each** bucket (0 / 1 / 2-3 / 4+) into one small model.

In [ ]:
N_ITER          = 5     # <- number of independent (tool-sample + finetune) iterations
TOOLS_PER_BUCKET = 2     # tools drawn from each param bucket per iteration
CAP_PER_TOOL     = 40
EPOCHS           = 1
EVAL_SUBSAMPLE   = 40
BATCH_SIZE       = 8     # safe on L4 (kernel + train subprocess share the GPU); raise to 16 if headroom, lower to 4 on OOM
RUN_TRAIN        = True
FORCE_RETRAIN    = False
print("param buckets available:", tools_dataframe(cat, FOCUS)["param_bucket"].value_counts().to_dict())

## 3 · Run the iterations\n*Resumable:* finished iterations are skipped; per-tool rows persist to `OUT_DIR/depth_results.json` after each iteration.

In [ ]:
RES = os.path.join(OUT_DIR, "depth_results.json")
rows = json.load(open(RES)) if (os.path.exists(RES) and not FORCE_RETRAIN) else []
done_iters = {r["iteration"] for r in rows}
if done_iters: log(f"loaded {len(done_iters)} finished iteration(s) from {RES}")

t_all = time.time()
for i in range(N_ITER):
    ckpt = os.path.join(OUT_DIR, f"picko_depth_iter{i}_best.pkl")
    if (i in done_iters) and os.path.exists(ckpt) and not FORCE_RETRAIN:
        log(f"iter {i}: skip (already done)"); continue
    try:
        names = sample_stratified(cat, FOCUS, TOOLS_PER_BUCKET, seed=i)
        log(f"=== start iter {i} · tools={names} ===")
        R = finetune_and_eval(cat, raw, tok, names, f"depth_iter{i}", OUT_DIR,
                              cap=CAP_PER_TOOL, epochs=EPOCHS, compact=False, token_aware=True,
                              eval_subsample=EVAL_SUBSAMPLE, run_train=RUN_TRAIN,
                              force_retrain=FORCE_RETRAIN, batch_size=BATCH_SIZE)
        new = []
        for tool, s in R["metrics"]["per_tool"].items():
            _, tot = cat.params_of(tool)
            new.append({"iteration": i, "tool": tool, "total_params": tot,
                        "param_bucket": param_bucket(tot), "n": s["n"],
                        "selection_acc": s["selection_acc"],
                        "args_exact_acc": s["args_exact_acc"], "param_f1": s["param_f1"]})
        rows = [r for r in rows if r["iteration"] != i] + new
        done_iters.add(i)
        json.dump(rows, open(RES, "w"), indent=2)   # persist each iteration
        log(f"=== done iter {i}: {len(new)} tools measured ===")
    except Exception as e:
        log(f"iter {i}: FAILED ({type(e).__name__}: {e}) — skipping; re-run to resume")

log(f"ALL ITERATIONS DONE in {time.time()-t_all:.0f}s · results={RES}")
depth = pd.DataFrame(rows)
print("collected", len(depth), "per-tool measurements across", depth["iteration"].nunique(), "iterations")
display(depth.head(12))

## 4 · Extraction accuracy per parameter bucket (mean ± std)

In [ ]:
# per-iteration bucket means first (paired within iteration), then mean/std across iterations
per_iter = (depth.groupby(["iteration","param_bucket"])[["args_exact_acc","param_f1"]]
            .mean().reset_index())
agg = (per_iter.groupby("param_bucket")
       .agg(args_mean=("args_exact_acc","mean"), args_std=("args_exact_acc","std"),
            pf1_mean=("param_f1","mean"), pf1_std=("param_f1","std"),
            n_iter=("iteration","nunique"))
       .reindex([b for b in PARAM_BUCKET_ORDER if b in per_iter["param_bucket"].values]))
display(agg.round(3))

x = np.arange(len(agg)); w = 0.38
fig, ax = plt.subplots(figsize=(8,4.5))
ax.bar(x-w/2, agg["args_mean"], w, yerr=agg["args_std"].fillna(0), capsize=4, color="#4C72B0", label="args_exact_acc")
ax.bar(x+w/2, agg["pf1_mean"], w, yerr=agg["pf1_std"].fillna(0), capsize=4, color="#DD8452", label="param_f1")
# overlay each iteration's bucket mean as points
for _, r in per_iter.iterrows():
    xi = list(agg.index).index(r["param_bucket"]) if r["param_bucket"] in list(agg.index) else None
    if xi is not None: ax.scatter(xi-w/2, r["args_exact_acc"], color="#243b57", s=14, zorder=3)
ax.set_xticks(x); ax.set_xticklabels(agg.index); ax.set_ylim(0,1)
ax.set_xlabel("# parameters (bucket)"); ax.set_ylabel("accuracy")
ax.set_title(f"Depth: parameter extraction vs #params ({int(agg['n_iter'].max())} iterations)"); ax.legend()
plt.tight_layout(); save_fig("depth_buckets"); plt.show()

## 5 · Per-tool scatter (all iterations)

In [ ]:
tool_mean = depth.groupby(["tool","total_params"])["args_exact_acc"].mean().reset_index()
plt.figure(figsize=(7.5,4.5))
plt.scatter(tool_mean["total_params"], tool_mean["args_exact_acc"], s=55, color="#4C72B0")
for _, r in tool_mean.iterrows():
    plt.annotate(r["tool"].split("_")[0], (r["total_params"], r["args_exact_acc"]), fontsize=7)
plt.xlabel("# parameters in tool"); plt.ylabel("mean args_exact_acc")
plt.title("Depth: per-tool extraction vs parameter count")
plt.tight_layout(); save_fig("depth_scatter"); plt.show()

## 6 · Read-out

Argument extraction is near-solved for **0–1 parameter** tools and **degrades for multi-parameter (4+)**
tools — the error bars show it's a consistent effect across independent tool samples, not one unlucky
model. This is where a small specialist model needs the most help (and where finetuning gains most).